# Lakeflow Connect: SharePoint Connector Demo

This notebook demonstrates how to use the **Databricks SharePoint Connector** (currently in Beta) to ingest various file types from Microsoft SharePoint into Delta tables with Unity Catalog governance. Make a copy of this notebook to run it yourself.

View our public documentation [here](https://docs.databricks.com/aws/en/ingestion/sharepoint). 

## What You'll Learn

* **Ingest unstructured PDFs** - Read PDF files and parse them using AI functions
* **Sync Excel files** - Load individual Excel files into Delta tables
* **Ingest CSV files** - Merge multiple structured files with the same schema
* **Automatic Incremental Ingestion using Lakeflow SDP** - Orchestrate all of your ingestion automatically using Lakeflow Spark Declarative Pipelines

## Prerequisites

* **Unity Catalog Connection**: You'll need to create a UC connection to your SharePoint environment. Please view our public documentation to learn how: [link to docs](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/sharepoint-source-setup-overview) 
* **Databricks Runtime**: 17.3 or above (required for SharePoint Connector)

## About the SharePoint Connector

The SharePoint Connector supports:
* Batch and streaming ingestion (Auto Loader, spark.read, COPY INTO)
* Structured (CSV, Excel), semi-structured (JSON), and unstructured (PDF, images) files
* Unity Catalog governance and security
* Automatic schema inference and evolution 

---

## 1. Ingest Unstructured PDFs and Parse with AI

This section demonstrates how to:
1. Read PDF files from SharePoint as binary files
2. Save them to a Delta table
3. Use SQL AI functions to parse and extract structured content from PDFs

In [0]:
# Set the default catalog and schema for this session
spark.sql("USE CATALOG dennis_schultz")
spark.sql("USE SCHEMA sharepoint_testing")

In [0]:
# Read all PDF files from SharePoint as binary files
# This is a batch read. For automatic incremental ingestion, view the cells at the bottom of the notebook to see how to use this connector in Lakeflow Spark Declarative Pipelines
pdf_df = (spark.read
    .format("binaryFile") # Use this format for unstructured data
    .option("databricks.connection", "brickfood_sharepoint_connection") # User provides the name of their connection
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pdf")  # Only ingest PDF files
    .load("https://databricks977.sharepoint.com/sites/brickfood-demo-site/Shared%20Documents/Forms/AllItems.aspx")
    .select("*", "_metadata")
)

# Save the PDF files to a Delta table for persistent storage
pdf_df.write \
    .mode("overwrite") \
    .saveAsTable("aircraft_maintenance_logs")

# Display the DataFrame to see the PDF files
display(pdf_df.limit(1))

### Parse PDFs using `ai_parse_document()`

When ingesting unstructured files from SharePoint (such as PDFs, Word documents, or PowerPoint files) using the standard SharePoint connector with binaryFile format, the file contents are stored as raw binary data.

To prepare these files for AI workloads—such as RAG, search, classification, or document understanding—you can easily parse the binary content into structured, queryable output by just applying `ai_parse_document()` onto the `content` column.

In [0]:
%sql
-- Parse PDF content using AI and extract structured information. 
SELECT
  *,
  ai_parse_document(content) AS parsed_content
FROM
  aircraft_maintenance_logs
LIMIT 5 -- limiting to only the first 5 PDFs

You can also use `ai_parse_document()` **within Lakeflow Spark Declarative Pipelines to enable incremental parsing**. As new files stream in from SharePoint, they are automatically parsed as your pipeline updates.

---

## 2. Sync Individual Excel Files to Delta Tables

This section demonstrates how to:
1. Read a specific Excel file from SharePoint using Python `spark.read()` or SQL `read_files()`

Excel files support options like `headerRows` and `dataAddress` to specify which sheet and range to read.

In [0]:
# Read a specific Excel file from SharePoint
excel_df = (spark.read
    .format("excel")
    .option("databricks.connection", "brickfood_sharepoint_connection")
    .option("headerRows", 1)  # First row contains headers
    .option("inferSchema", True)  # Automatically infer column types
    .option("dataAddress", "Sheet1!A1:Y200")
    .load("https://databricks977.sharepoint.com/sites/brickfood-demo-site/Sample%20Files/Example%20Excel%20Files/sales_data_sample.xlsx")
)

# Save the Excel data to a Delta table
excel_df.write \
    .mode("overwrite") \
    .saveAsTable("brickfood_excel_report")

# Display the Excel data
display(excel_df)

In [0]:
%sql
-- Create a table from an Excel file using SQL
CREATE OR REPLACE TABLE brickfood_excel_report_sql AS
SELECT * FROM read_files(
  "https://databricks977.sharepoint.com/sites/brickfood-demo-site/Sample%20Files/Example%20Excel%20Files/sales_data_sample.xlsx",
  `databricks.connection` => "brickfood_sharepoint_connection",
  format => "excel",
  headerRows => 1,
  inferSchema => true,
  schemaEvolutionMode => "none"
);


-- Query the newly created table
SELECT * FROM brickfood_excel_report_sql LIMIT 10

---

## 3. Batch Ingest Multiple CSV Files with Same Schema

This section demonstrates how to:
1. Read multiple CSV files from a SharePoint folder
2. Merge them into an existing Delta table
3. Use COPY INTO for idempotent incremental loading

This is useful when you have multiple files with the same structure that need to be consolidated into a single table.

In [0]:
# Read all CSV files from a SharePoint folder
csv_df = (spark.read
    .format("csv")
    .option("databricks.connection", "brickfood_sharepoint_connection")
    .option("pathGlobFilter", "*.csv")  # Only read CSV files
    .option("recursiveFileLookup", True)  # Search subdirectories
    .option("inferSchema", True)  # Automatically infer column types
    .option("header", True)  # First row contains headers
    .load("https://databricks977.sharepoint.com/sites/brickfood-demo-site/Sample%20Files/Sample%20CSV%20with%20Same%20Schema")
)

# Create or replace the Delta table with CSV data
csv_df.write \
    .mode("overwrite") \
    .saveAsTable("brickfood_csv_data")

print("✅ CSV files saved to Delta table: brickfood_csv_data")

# Display the combined CSV data
display(csv_df)

### Incremental Ingestion with COPY INTO

`COPY INTO` provides idempotent incremental loading - it automatically tracks which files have been processed and only loads new files. This is perfect for ongoing data ingestion where new CSV files are added to SharePoint over time.

In [0]:
%sql
-- Create the table if it doesn't exist
CREATE TABLE IF NOT EXISTS sharepoint_sample_csv_incremental;

-- Incrementally ingest new CSV files (only processes files not yet loaded)
COPY INTO sharepoint_sample_csv_incremental
  FROM "https://databricks977.sharepoint.com/sites/brickfood-demo-site/Sample%20Files/Sample%20CSV%20with%20Same%20Schema"
  FILEFORMAT = CSV
  PATTERN = '*.csv'
  FORMAT_OPTIONS('header' = 'true', 'inferSchema' = 'true', 'databricks.connection' = "brickfood_sharepoint_connection")
  COPY_OPTIONS ('mergeSchema' = 'true');

-- Query the incrementally loaded data
SELECT * FROM sharepoint_sample_csv_incremental LIMIT 10

---

## 4. Automatic, Incremental Ingestion using Lakeflow Spark Declarative Pipelines

While the previous examples showed batch ingestion, **Lakeflow Spark Declarative Pipelines** enable you to orchestrate **automatic, incremental ingestion** from SharePoint. As new files land in a SharePoint directory (or as files update), the pipeline automatically detects and ingests them.

### Key Benefits

* **Automatic file detection** - New or updated files are automatically discovered and processed
* **Incremental processing** - Only new data is processed, not the entire dataset
* **Schema evolution** - Automatically adapts to schema changes in your files
* **Orchestration** - Built-in scheduling and dependency management
* **Data quality** - Expectations and monitoring built into the pipeline

### Use Cases

* **Streaming tables** - For continuous ingestion of PDFs, CSVs, or other files as they arrive
* **Materialized views** - For specific Excel files that update periodically (e.g., monthly reports)

The examples below show how to define pipeline tables using both Python and SQL syntax.

### Lakeflow Spark Declarative Pipelines

Define streaming tables and materialized views using SQL syntax. Use `CREATE OR REFRESH STREAMING TABLE` for continuous ingestion and `CREATE OR REFRESH MATERIALIZED VIEW` for periodically refreshed data.

NOTE: The below code snippet examples must be copied over to an SDP pipeline in order to be deployed. View our Lakeflow Spark Declarative Pipelines Documentation to learn more on how to deploy an SDP pipeline.

NOTE: Users must set their pipeline channel to "PREVIEW" (default is "CURRENT") in order to use this feature in SDP. They can edit it via the pipeline settings --> JSON tab


In [0]:
%sql
-- NOTE: You must copy the code into an SDP pipeline to deploy. View our documentation to learn more.

-- Incrementally ingest new PDF files as they arrive
CREATE OR REFRESH STREAMING TABLE aircraft_maintenance_pdfs_streaming_sql
AS SELECT * FROM STREAM read_files(
  "https://databricks977.sharepoint.com/sites/brickfood-demo-site/Shared%20Documents/Forms/AllItems.aspx",
  format => "binaryFile",
  `databricks.connection` => "brickfood_sharepoint_connection_v2",
  pathGlobFilter => "*.pdf"
);

In [0]:
%sql
-- NOTE: You must copy the code into an SDP pipeline to deploy. View our documentation to learn more.

-- Incrementally ingest CSV files with automatic schema inference and evolution
CREATE OR REFRESH STREAMING TABLE sharepoint_sample_csv_streaming_sql
AS SELECT * FROM STREAM read_files(
  "https://databricks977.sharepoint.com/sites/brickfood-demo-site/Sample%20Files/Sample%20CSV%20with%20Same%20Schema",
  format => "csv",
  `databricks.connection` => "brickfood_sharepoint_connection_v2",
  pathGlobFilter => "*.csv",
  header => "true"
);

In [0]:
%sql
-- NOTE: You must copy the code into an SDP pipeline to deploy. View our documentation to learn more.

-- Read a specific Excel file from SharePoint in a materialized view
-- This will automatically refresh when the Excel file is updated
CREATE OR REFRESH MATERIALIZED VIEW sharepoint_sample_excel_materialized_sql
AS SELECT * FROM read_files(
  "https://databricks977.sharepoint.com/sites/brickfood-demo-site/Sample%20Files/Example%20Excel%20Files/sales_data_sample.xlsx",
  `databricks.connection` => "brickfood_sharepoint_connection_v2",
  format => "excel",
  headerRows => 1,
  `cloudFiles.schemaEvolutionMode` => "none"
);

---

## Next Steps and Resources

### What You've Learned

You've successfully explored:
* ✅ Batch ingestion of PDFs, Excel, and CSV files from SharePoint
* ✅ AI-powered document parsing with `ai_parse_document()`
* ✅ Incremental loading with COPY INTO
* ✅ Automatic, orchestrated ingestion with Lakeflow Spark Declarative Pipelines

### Additional AI Functions to Explore

Once you've parsed your documents, explore these AI functions for further analysis:
* `ai_summarize()` - Generate summaries of text content
* `ai_extract()` - Extract specific entities from documents
* `ai_classify()` - Classify documents into categories
* `ai_analyze_sentiment()` - Analyze sentiment in text
* `ai_query()` - Query foundation models for custom analysis

### Resources

* **Documentation**: [SharePoint Connector Guide](https://docs.databricks.com/aws/en/ingestion/sharepoint)
* **Lakeflow Pipelines**: [Spark Declarative Pipelines Documentation](https://www.databricks.com/product/data-engineering/spark-declarative-pipelines)
* **AI Functions**: [SQL AI Functions Reference](https://docs.databricks.com/aws/en/large-language-models/ai-functions)

### Questions?

For internal Databricks employees: Provide feedback and questions in [#lakeflow-connect](https://databricks.enterprise.slack.com/archives/C05HQQEAZ0D)